<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2024 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 CodeGemma 和 KerasNLP 進行 AI 輔助編程

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/codegemma/code_assist_keras"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/docs/codegemma/code_assist_keras.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemma/cookbook/blob/main/docs/codegemma/code_assist_keras.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemmma%2Fcookbook%2Fmain%2Fdocs%2Fcodegemma%2Fcode_assist_keras.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/docs/codegemma/code_assist_keras.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

## 概述

CodeGemma 是 Gemma 的變體，針對編碼任務進行了微調。本教學以 [Keras CodeGemma 快速入門](https://colab.research.google.com/drive/11Va7W2Yl12JUnfx9YDJmy90kBu1i4rtw) 為基礎，並向您展示 CodeGemma 協助您完成程式設計任務的更多方法。

## 設定

### 訪問CodeGemma

要完成本教學，您首先需要完成 [Gemma 設定](https://ai.google.dev/gemma/docs/setup) 中的設定說明。 Gemma 設定說明向您展示如何執行以下操作：
* 在 [kaggle.com](https://kaggle.com){:.external} 上造訪Gemma。
* 選擇具有足夠資源執行Gemma 7B 模型的Colab runtime。
* 產生並設定 Kaggle 使用者名稱和 API 金鑰。

完成 Gemma 設定後，請前往下一部分，您將為 Colab 環境設定環境變數。

### 選擇runtime

要執行CodeGemma 7B 型號，您需要有付費Colab Pro 計劃，該計劃提供帶有 A100 GPU 的runtime。
1. 在Colab視窗的右上角，選擇&#9662; （**附加連線選項**）。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **A100 GPU**。

### 設定您的 API 金鑰

To use Gemma, you must provide your Kaggle username and a Kaggle API key.

若要產生 Kaggle API 金鑰，請前往 Kaggle 使用者個人資料的 **帳戶** 選項卡，然後選擇 **建立新 token**。這將觸發包含您的 API 憑證的 `kaggle.json` 檔案的下載。

在 Colab 中，選擇左側窗格中的 **Secrets** (🔑)，然後新增您的 Kaggle 使用者名稱和 Kaggle API 金鑰。將您的使用者名稱儲存在名稱`KAGGLE_USERNAME` 下，將您的API 金鑰儲存在名稱`KAGGLE_KEY` 下。

### 設定環境變數

設定`KAGGLE_USERNAME` 和`KAGGLE_KEY` 的環境變數。

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### 安裝依賴項

In [ ]:
!pip install -q -U keras-nlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.4/508.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 51.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 589.8/589.8 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 59.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 42.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 38.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.15.1 requires tensorflow<2.16,>=2.15, but you have tensorflow 2.16.1 which is incompatibl

### 選擇後端

Keras 是高級、多framework 深度學習API，設計簡單易用。使用Keras 3，您可以在三個後端之一上執行工作流程：TensorFlow、JAX 或PyTorch。
在本教學中，為 JAX 設定後端。

In [ ]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "tensorflow" or "torch".

### 導入包

導入 Keras 和 KerasNLP。

In [ ]:
import keras_nlp
import keras

# Run at half precision.
keras.config.set_floatx("bfloat16")

## CodeGemma 7B 模型範例

本節介紹使用預先訓練的 7B CodeGemma 模型來幫助完成編碼任務的範例。

### 載入模型

KerasNLP 使用 [`GemmaCausalLM`](https://keras.io/api/keras_nlp/models/gemma/gemma_causal_lm/){:.external}（用於因果語言建模的端對端 Gemma 模型）提供所有三種 CodeGemma 變體（2B 和 7B 實現預訓練 (PT) 和 IT)）的指令 IT)。因果語言模型根據前一個 tokens 預測下一個 token。

對於本範例，使用 [`from_preset`](https://keras.io/api/keras_nlp/models/gemma/gemma_causal_lm/#frompreset-method){:.external} 方法載入 `code_gemma_7b_en` 模型。


In [ ]:
gemma_lm_7b = keras_nlp.models.GemmaCausalLM.from_preset("code_gemma_7b_en")

100%|██████████| 556/556 [00:00<00:00, 790kB/s]
100%|██████████| 15.9G/15.9G [02:39<00:00, 107MB/s]
100%|██████████| 401/401 [00:00<00:00, 587kB/s]
100%|██████████| 4.04M/4.04M [00:00<00:00, 16.4MB/s]


In [ ]:
gemma_lm_7b.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tokenizer (type)                                   ┃                                             Vocab # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                   │                                             256,000 │
└────────────────────────────────────────────────────┴─────────────────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 3072)        │   8,537,680,896 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     786,432,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 8,537,680,896 (15.90 GB)

 Trainable params: 8,537,680,896 (15.90 GB)

 Non-trainable params: 0 (0.00 B)

`from_preset` 方法根據預設的架構和權重實例化模型。

### 使用多行 FIM 完成程式碼

PT CodeGemma 模型接受程式碼填充任務的訓練。本節展示使用 CodeGemma 的多行中間填充 (FIM) 功能根據周圍上下文在指定遊標位置自動填充程式碼的範例。

第一步，定義常數和 prompt 格式化輔助函數。

In [ ]:
# Formatting control tokens to specify cursor location
BEFORE_CURSOR = "<|fim_prefix|>"
AFTER_CURSOR = "<|fim_suffix|>"
AT_CURSOR = "<|fim_middle|>"
FILE_SEPARATOR = "<|file_separator|>"

# Define model stop tokens
END_TOKEN = gemma_lm_7b.preprocessor.tokenizer.end_token
stop_tokens = (BEFORE_CURSOR, AFTER_CURSOR, AT_CURSOR, FILE_SEPARATOR, END_TOKEN)
stop_token_ids = tuple(gemma_lm_7b.preprocessor.tokenizer.token_to_id(x) for x in stop_tokens)

def format_completion_prompt(before, after):
    return f"{BEFORE_CURSOR}{before}{AFTER_CURSOR}{after}{AT_CURSOR}"

#### 範例 1 - 插入缺失條件

如果`n=1`，下方產生斐波那契數列的範例程式碼將無法正確執行：
```python
def fibonacci(n: int) -> int:
  if n == 0:
    return 0
  # The cursor is right before the e in the following line
  else:
    return fibonacci(n - 1) + fibonacci(n - 2)
```

假設遊標位於第4行開頭（`else`子句所在），則遊標前後的內容為：

In [ ]:
before = """def fibonacci(n: int) -> int:\n  if n == 0:\n    return 0\n""" # Mind the spaces!
after = """\n  else:\n    return fibonacci(n - 1) + fibonacci(n-2)\n"""

In [ ]:
prompt = format_completion_prompt(before, after)
print(prompt)

<|fim_prefix|>def fibonacci(n: int) -> int:
  if n == 0:
    return 0
<|fim_suffix|>
  else:
    return fibonacci(n - 1) + fibonacci(n-2)
<|fim_middle|>


Run the prompt.

In [ ]:
print(gemma_lm_7b.generate(prompt, stop_token_ids=stop_token_ids, max_length=128))

<|fim_prefix|>def fibonacci(n: int) -> int:
  if n == 0:
    return 0
<|fim_suffix|>
  else:
    return fibonacci(n - 1) + fibonacci(n-2)
<|fim_middle|>elif n == 1:
    return 1<|file_separator|>


模型會在遊標位置插入`n=1` 的正確`elif` 條件。

#### 範例2-完整的 DFS 遍歷演算法

深度優先搜尋 (DFS) 樹遍歷演算法的自動完成程式碼。

In [ ]:
before = """void dfs(node* root) {
  if (root->left) {
    dfs(root->left);
  }"""
after = """\nprintf("%d", root->value);
}"""

In [ ]:
prompt = format_completion_prompt(before, after)
print(prompt)

<|fim_prefix|>void dfs(node* root) {
  if (root->left) {
    dfs(root->left);
  }<|fim_suffix|>
printf("%d", root->value);
}<|fim_middle|>


Run the prompt.

In [ ]:
print(gemma_lm_7b.generate(prompt, stop_token_ids=stop_token_ids, max_length=128))

<|fim_prefix|>void dfs(node* root) {
  if (root->left) {
    dfs(root->left);
  }<|fim_suffix|>
printf("%d", root->value);
}<|fim_middle|>
  if (root->right) {
    dfs(root->right);
  }<|file_separator|>


### 程式碼生成

除了程式碼填充之外，CodeGemma 7B PT is 模型還在自然語言語料庫上進行了訓練。您可以使用它prompt模型來產生程式碼。

In [ ]:
generation_prompt= """Write a rust function to identify non-prime numbers.
Examples:
>>> is_not_prime(2)
False
>>> is_not_prime(10)
True
pub fn is_not_prime(n: i32) -> bool {"""

In [ ]:
print(gemma_lm_7b.generate(generation_prompt, max_length=500))

Write a rust function to identify non-prime numbers.
Examples:
>>> is_not_prime(2)
False
>>> is_not_prime(10)
True
pub fn is_not_prime(n: i32) -> bool {
    if n <= 1 {
        return true;
    }
    for i in 2..n {
        if n % i == 0 {
            return true;
        }
    }
    false
}



## 7B IT 模型範例

本節使用 CodeGemma 7B 指令調整模型來執行更進階的編碼任務。 CodeGemma 7B IT 模型源自 CodeGemma 7B PT 模型，透過程式碼上的監督 fine-tuning 以及帶有人類回饋的強化學習。本節介紹使用此模型進行開放式產生的範例。
注意：如果您正在 Colab 中完成本教學，並且已經從上面加載了 2B 模型，請轉到 **執行時** > **斷開並刪除 runtime** 並重新連接到新的 runtime，重新啟動 Colab 執行時。這可以釋放記憶體並防止記憶體不足 (OOM) 問題。連接到新的 runtime 後，請[從此處](#set_environment_variables) 重新執行設定步驟，然後再繼續。

### 載入 IT 模型

使用`from_preset`方法載入`code_gemma_instruct_7b_en`模型。

In [ ]:
gemma_lm_7b_it = keras_nlp.models.GemmaCausalLM.from_preset("code_gemma_instruct_7b_en")
gemma_lm_7b_it.summary()

100%|██████████| 556/556 [00:00<00:00, 754kB/s]
100%|██████████| 15.9G/15.9G [03:18<00:00, 86.2MB/s]
100%|██████████| 401/401 [00:00<00:00, 593kB/s]
100%|██████████| 4.04M/4.04M [00:00<00:00, 16.8MB/s]


Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tokenizer (type)                                   ┃                                             Vocab # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                   │                                             256,000 │
└────────────────────────────────────────────────────┴─────────────────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 3072)        │   8,537,680,896 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     786,432,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 8,537,680,896 (15.90 GB)

 Trainable params: 8,537,680,896 (15.90 GB)

 Non-trainable params: 0 (0.00 B)

IT 模型使用特定的格式化程式進行訓練，該格式化程式使用額外資訊來註釋所有指令調整範例，以指示角色並描繪對話中的輪次。
第一步，定義常數和 prompt 格式化輔助函數。

In [ ]:
# Formatting control tokens for instruction tuning
START_OF_TURN_USER = "<start_of_turn>user"
END_OF_TURN = "<end_of_turn>"
START_OF_TURN_MODEL = "<start_of_turn>model"

# Formatting helper function
def format_instruction_prompt(context):
    return f"{START_OF_TURN_USER}\n{context}{END_OF_TURN}\n{START_OF_TURN_MODEL}\n"

### 程式碼翻譯


In [ ]:
context1 = """
You are an experienced C and Python programmer. Convert the following Python code into C.
```python
def factorial(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```\n"""

格式化prompt。

In [ ]:
prompt1 = format_instruction_prompt(context1)
print(prompt1)

<start_of_turn>user

You are an experienced C and Python programmer. Convert the following Python code into C.
```python
def factorial(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```
<end_of_turn>
<start_of_turn>model



Run the prompt.

In [ ]:
print(gemma_lm_7b_it.generate(prompt1, max_length=500))

<start_of_turn>user

You are an experienced C and Python programmer. Convert the following Python code into C.
```python
def factorial(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```
<end_of_turn>
<start_of_turn>model
Here is the C code equivalent of the Python code:

```c
int factorial(int n) {
  int result = 1;
  for (int i = 2; i <= n; i++) {
    result *= i;
  }
  return result;
}
```

Here is a breakdown of the changes:

* The function is declared with the `int` return type, as in Python.
* The `for` loop is converted to a `for` loop with an `int` variable `i` initialized to 2 and incremented by 1 in each iteration.
* The `range` function is replaced with a simple loop that iterates from 2 to `n` (inclusive).
* The `result *= i` statement is used to multiply `result` by `i` in each iteration.
* The `return` statement is used to return the final value of `result`.


### 程式碼漏洞檢測


In [ ]:
context2 = """
You are an experienced C++ programmer hunting for vulnerable code. Is the following code vulnerable? Explain your reasoning.
```cpp
int i;
unsigned int numWidgets;
Widget **WidgetList;

numWidgets = GetUntrustedSizeValue();
if ((numWidgets == 0) || (numWidgets > MAX_NUM_WIDGETS)) {
    ExitError("Incorrect number of widgets requested!");
}
WidgetList = (Widget **) malloc(numWidgets * sizeof(Widget *));
printf("WidgetList ptr=%p\n", WidgetList);
for (i = 0; i < numWidgets; i++) {
    WidgetList[i] = InitializeWidget();
}
WidgetList[numWidgets] = NULL;
showWidgets(WidgetList);
```\n"""

格式化prompt。

In [ ]:
prompt2 = format_instruction_prompt(context2)
print(prompt2)

<start_of_turn>user

You are an experienced C++ programmer hunting for vulnerable code. Is the following code vulnerable? Explain your reasoning.
```cpp
int i;
unsigned int numWidgets;
Widget **WidgetList;

numWidgets = GetUntrustedSizeValue();
if ((numWidgets == 0) || (numWidgets > MAX_NUM_WIDGETS)) {
    ExitError("Incorrect number of widgets requested!");
}
WidgetList = (Widget **) malloc(numWidgets * sizeof(Widget *));
printf("WidgetList ptr=%p
", WidgetList);
for (i = 0; i < numWidgets; i++) {
    WidgetList[i] = InitializeWidget();
}
WidgetList[numWidgets] = NULL;
showWidgets(WidgetList);
```
<end_of_turn>
<start_of_turn>model



In [ ]:
print(gemma_lm_7b_it.generate(prompt2, max_length=1000))

<start_of_turn>user

You are an experienced C++ programmer hunting for vulnerable code. Is the following code vulnerable? Explain your reasoning.
```cpp
int i;
unsigned int numWidgets;
Widget **WidgetList;

numWidgets = GetUntrustedSizeValue();
if ((numWidgets == 0) || (numWidgets > MAX_NUM_WIDGETS)) {
    ExitError("Incorrect number of widgets requested!");
}
WidgetList = (Widget **) malloc(numWidgets * sizeof(Widget *));
printf("WidgetList ptr=%p
", WidgetList);
for (i = 0; i < numWidgets; i++) {
    WidgetList[i] = InitializeWidget();
}
WidgetList[numWidgets] = NULL;
showWidgets(WidgetList);
```
<end_of_turn>
<start_of_turn>model
Yes, the code is vulnerable to a memory access error.

**Reasoning:**

* The code allocates memory for `WidgetList` using `malloc` based on the value of `numWidgets`.
* However, the loop iterates from `0` to `numWidgets`, which is one element beyond the allocated memory.
* This means that accessing `WidgetList[numWidgets]` will result in a memory access err

該模型檢測程式碼中的潛在漏洞並提供程式碼變更以緩解該漏洞。

## 概括

本教學引導您使用 CodeGemma 完成各種編碼任務。要了解有關CodeGemma的更多資訊：
* 有關CodeGemma 型號的技術規格，請參閱[CodeGemma 型號卡](https://ai.google.dev/gemma/docs/codegemma/model_card)。
* 了解有關如何在 VertexAI 中使用 CodeGemma 的更多資訊 [此處](https://colab.sandbox.google.com/github/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/model_garden/model_garden_codegemma_deployment_on_vertex.ipynb)。
* 查看 [Keras CodeGemma 快速入門](https://ai.google.dev/gemma/docs/codegemma/keras_quickstart)。